Cell 1. 경로 설정

In [1]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.path_utils import find_project_root

PROJECT_ROOT = find_project_root("RFP-RAG-Extractor")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

EXTRACTED_DIR = PROCESSED_DIR / "extracted"
CLEANED_DIR = PROCESSED_DIR / "cleaned"

CHUNK_DIR = DATA_DIR / "chunks" / "section"
SECTION_CHUNK_PATH = CHUNK_DIR / "section_chunks.jsonl"

DATA_LIST_PATH = RAW_DIR / "data_list.csv"

print("현재 작업 위치:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("DATA_LIST_PATH:", DATA_LIST_PATH)
print("SECTION_CHUNK_PATH:", SECTION_CHUNK_PATH)

현재 작업 위치: /home/user1/RFP-RAG-Extractor/notebooks
PROJECT_ROOT: /home/user1/RFP-RAG-Extractor
RAW_DIR: /home/user1/RFP-RAG-Extractor/data/raw
DATA_LIST_PATH: /home/user1/RFP-RAG-Extractor/data/raw/data_list.csv
SECTION_CHUNK_PATH: /home/user1/RFP-RAG-Extractor/data/chunks/section/section_chunks.jsonl


Cell 2. 모듈 import

In [2]:
from src.extractors import extract_text_by_file_type
from src.utils.text_cleaner import clean_extracted_text
from src.chunking.section_chunker import create_section_chunks
from src.utils.file_utils import save_jsonl

Cell 3. data_list.csv 로드

In [3]:
data_list = pd.read_csv(DATA_LIST_PATH)

print("문서 수:", len(data_list))
data_list.head()

문서 수: 100


,공고 번호,공고 차수,사업명,사업 금액,발주 기관,공개 일자,입찰 참여 시작일,입찰 참여 마감일,사업 요약,파일형식,파일명,텍스트
0,20241001798,0.0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,130000000.0,한영대학,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00,- 한영대학교 특성화 맞춤형 교육환경 구축을 위해 트랙운영 학사정보시스템을 고도화한...,hwp,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,\n \n2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 ...
1,20241002912,0.0,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,129300000.0,한국연구재단,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00,- 사업 개요: 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선\n...,hwp,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,\r\n \r\n \r\n \r\n제 안 요 청 서\r\n[ 2024년 대학 ...
2,20240827859,0.0,EIP3.0 고압가스 안전관리 시스템 구축 용역,40000000.0,한국생산기술연구원,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00,- 사업 개요: EIP3.0 고압가스 안전관리 시스템 구축 용역\n- 추진배경: 안...,hwp,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,\r\n \r\nEIP3.0 고압가스 안전관리\r\n시스템 구축 용역\...
3,20240430918,0.0,도시계획위원회 통합관리시스템 구축용역,150000000.0,인천광역시,2024-04-18 16:26:32,2024-05-02 10:00:00,2024-05-09 16:00:00,- 사업명: 도시계획위원회 통합관리시스템 구축 용역\n- 용역개요: 도시계획위원회와...,hwp,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,\r\n \r\n \r\n도시계획위원회 통합관리시스템 구축\r\n제 안 요 청...
4,20240430896,0.0,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),900000000.0,경상북도 봉화군,2024-04-18 16:33:28,2024-04-26 09:00:00,2024-04-30 17:00:00,- 사업명: 봉화군 재난통합관리시스템 고도화 사업\n- 사업개요: 공동수급(공동이행...,hwp,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,\r\n \r\n \r\n제안요청서\r\n \r\n사 업 명\r\n봉화...


Cell 4. 원본 파일 경로 확인

In [4]:
import re
import unicodedata
from pathlib import Path
import pandas as pd


def normalize_file_name(name: str) -> str:
    """
    파일명 비교용 정규화 함수입니다.

    raw 파일명이 NFD, data_list.csv 파일명이 NFC인 경우를 맞추기 위해
    Unicode NFC 정규화를 적용합니다.

    처리:
    - Unicode NFC 정규화
    - 소문자화
    - 앞뒤 공백 제거
    - 모든 공백 제거
    - 일부 기호 통일
    """
    if name is None or pd.isna(name):
        return ""

    name = str(name).strip()

    # 핵심: 한글 자모 분리/조합 문제 해결
    name = unicodedata.normalize("NFC", name)

    name = name.lower()

    # 공백 제거
    name = re.sub(r"\s+", "", name)

    # 비슷한 기호 통일
    replacements = {
        "－": "-",
        "–": "-",
        "—": "-",
        "＿": "_",
        "（": "(",
        "）": ")",
        "㈜": "(주)",
        "（주）": "(주)",
    }

    for src, dst in replacements.items():
        name = name.replace(src, dst)

    return name


def find_raw_file(file_name: str) -> Path | None:
    """
    data/raw 하위에서 data_list.csv의 파일명에 해당하는 원본 파일을 찾습니다.

    매칭 순서:
    1. 원본 파일명 완전 일치
    2. Unicode/공백/기호 정규화 후 파일명 전체 일치
    3. stem 정규화 후 일치
    4. stem 포함 관계 + 확장자 동일
    """
    if file_name is None or pd.isna(file_name):
        return None

    file_name = str(file_name).strip()

    raw_files = [p for p in RAW_DIR.rglob("*") if p.is_file()]

    target_name_norm = normalize_file_name(file_name)
    target_stem_norm = normalize_file_name(Path(file_name).stem)
    target_suffix = Path(file_name).suffix.lower()

    # 1. 파일명 완전 일치
    for path in raw_files:
        if path.name == file_name:
            return path

    # 2. 정규화 후 파일명 전체 일치
    for path in raw_files:
        if normalize_file_name(path.name) == target_name_norm:
            return path

    # 3. 정규화된 stem 일치
    for path in raw_files:
        if normalize_file_name(path.stem) == target_stem_norm:
            return path

    # 4. stem 포함 관계 + 확장자 동일
    for path in raw_files:
        path_stem_norm = normalize_file_name(path.stem)
        path_suffix = path.suffix.lower()

        suffix_ok = (
            not target_suffix
            or path_suffix == target_suffix
        )

        if suffix_ok and target_stem_norm:
            if target_stem_norm in path_stem_norm or path_stem_norm in target_stem_norm:
                return path

    return None

In [5]:
# 파일 경로 매칭
data_list["file_path"] = data_list["파일명"].apply(find_raw_file)
data_list["file_exists"] = data_list["file_path"].apply(
    lambda x: x is not None and Path(x).exists()
)

print(data_list["file_exists"].value_counts(dropna=False))

data_list[
    ["공고 번호", "사업명", "파일형식", "파일명", "file_path", "file_exists"]
].head()

file_exists
True    100
Name: count, dtype: int64


,공고 번호,사업명,파일형식,파일명,file_path,file_exists
0,20241001798,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,hwp,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,/home/user1/RFP-RAG-Extractor/data/raw/한영ᄃ...,True
1,20241002912,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,hwp,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,/home/user1/RFP-RAG-Extractor/data/raw/한국ᄋ...,True
2,20240827859,EIP3.0 고압가스 안전관리 시스템 구축 용역,hwp,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,/home/user1/RFP-RAG-Extractor/data/raw/한국ᄉ...,True
3,20240430918,도시계획위원회 통합관리시스템 구축용역,hwp,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,/home/user1/RFP-RAG-Extractor/data/raw/인천ᄀ...,True
4,20240430896,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),hwp,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,/home/user1/RFP-RAG-Extractor/data/raw/경상ᄇ...,True


In [6]:
# 파일 경로 매칭 실패
missing_df = data_list[~data_list["file_exists"]].copy()

print("못 찾은 파일 수:", len(missing_df))

missing_df[
    ["공고 번호", "사업명", "파일형식", "파일명"]
].head(30)

못 찾은 파일 수: 0


,공고 번호,사업명,파일형식,파일명


Cell 5. doc_id 생성

In [7]:
def make_doc_id(row) -> str:
    notice_no = row.get("공고 번호")

    if pd.notna(notice_no) and str(notice_no).strip():
        return str(notice_no).strip()

    return f"DOC_{int(row.name):04d}"


data_list["doc_id"] = data_list.apply(make_doc_id, axis=1)

data_list[["doc_id", "공고 번호", "사업명", "파일명"]].head()

,doc_id,공고 번호,사업명,파일명
0,20241001798,20241001798,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
1,20241002912,20241002912,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp
2,20240827859,20240827859,EIP3.0 고압가스 안전관리 시스템 구축 용역,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
3,20240430918,20240430918,도시계획위원회 통합관리시스템 구축용역,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp
4,20240430896,20240430896,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp


Cell 6. 텍스트 추출 + 정제 + 청킹 실행

In [8]:
all_chunks = []
process_logs = []

for idx, row in tqdm(data_list.iterrows(), total=len(data_list), desc="Extract/Clean/Chunk"):
    doc_id = row["doc_id"]
    file_path = row["file_path"]

    if file_path is None or not Path(file_path).exists():
        process_logs.append({
            "doc_id": doc_id,
            "file_name": row.get("파일명"),
            "status": "file_not_found",
            "error": "file_path not found"
        })
        continue

    try:
        extracted = extract_text_by_file_type(file_path)

        raw_text = extracted.get("text", "")
        clean_text = clean_extracted_text(raw_text)

        # 추출/정제 텍스트 저장
        extracted_path = EXTRACTED_DIR / f"{doc_id}.txt"
        cleaned_path = CLEANED_DIR / f"{doc_id}.txt"

        extracted_path.parent.mkdir(parents=True, exist_ok=True)
        cleaned_path.parent.mkdir(parents=True, exist_ok=True)

        extracted_path.write_text(raw_text, encoding="utf-8")
        cleaned_path.write_text(clean_text, encoding="utf-8")

        chunks = create_section_chunks(
            doc_id=doc_id,
            text=clean_text,
            file_name=row.get("파일명", ""),
            file_type=row.get("파일형식", ""),
            project_name=row.get("사업명", ""),
            organization=row.get("발주 기관", ""),
            max_chars=3000,
            overlap_chars=300,
            min_chars=100
        )

        all_chunks.extend(chunks)

        process_logs.append({
            "doc_id": doc_id,
            "file_name": row.get("파일명"),
            "file_type": row.get("파일형식"),
            "status": "success",
            "raw_text_len": len(raw_text),
            "clean_text_len": len(clean_text),
            "num_chunks": len(chunks),
            "error": ""
        })

    except Exception as e:
        process_logs.append({
            "doc_id": doc_id,
            "file_name": row.get("파일명"),
            "file_type": row.get("파일형식"),
            "status": "failed",
            "raw_text_len": 0,
            "clean_text_len": 0,
            "num_chunks": 0,
            "error": str(e)
        })

print("총 청크 수:", len(all_chunks))

Extract/Clean/Chunk:   0%|          | 0/100 [00:00<?, ?it/s]

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict

총 청크 수: 14415


Cell 7. 청크 저장

In [9]:
save_jsonl(all_chunks, SECTION_CHUNK_PATH)

print("section_chunks 저장 완료:", SECTION_CHUNK_PATH)
print("총 청크 수:", len(all_chunks))

section_chunks 저장 완료: /home/user1/RFP-RAG-Extractor/data/chunks/section/section_chunks.jsonl
총 청크 수: 14415


Cell 8. 처리 로그 저장

In [10]:
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

process_log_df = pd.DataFrame(process_logs)
process_log_path = LOG_DIR / "extract_clean_chunk_log.csv"

process_log_df.to_csv(process_log_path, index=False, encoding="utf-8-sig")

print("로그 저장:", process_log_path)
process_log_df["status"].value_counts()

로그 저장: /home/user1/RFP-RAG-Extractor/logs/extract_clean_chunk_log.csv


status
success    100
Name: count, dtype: int64

Cell 9. 청크 품질 확인

In [11]:
chunk_df = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "file_type": chunk.get("file_type"),
        "project_name": chunk.get("project_name"),
        "section_title": chunk.get("section_title"),
        "text_len": len(chunk.get("text", ""))
    }
    for chunk in all_chunks
])

chunk_df.describe()

,text_len
count,14415.000000
mean,380.335831
std,459.127526
min,100.000000
25%,136.000000
50%,214.000000
75%,405.000000
max,3000.000000


In [12]:
print("문서별 청크 수")
chunk_df.groupby("doc_id")["chunk_id"].count().describe()

문서별 청크 수


count    100.00000
mean     144.15000
std       63.24162
min       51.00000
25%      105.00000
50%      130.00000
75%      165.25000
max      481.00000
Name: chunk_id, dtype: float64

In [13]:
print("너무 짧은 청크")
chunk_df[chunk_df["text_len"] < 100].head()

너무 짧은 청크


,chunk_id,doc_id,file_type,project_name,section_title,text_len


In [14]:
print("너무 긴 청크")
chunk_df[chunk_df["text_len"] > 3000].head()

너무 긴 청크


,chunk_id,doc_id,file_type,project_name,section_title,text_len
